In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [6]:
df_new = pd.read_csv('OctNov_CarData.csv')
df_old = pd.read_csv('total_call_data.csv')
df_new.head()

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\2363734591.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old = pd.read_csv('total_call_data.csv')


,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
0,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
1,9f5be247-1027-4443-bfe1-86a740f85b8d,NaN,FarmworkerMain,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
2,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-10-01 00:25:00,NaN,NaN,NaN,0
3,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,FarmworkerMain,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
4,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,FarmworkerMainMenu,2025-10-01 00:25:00,NaN,NaN,NaN,0


In [8]:
df_new['Queue Name'].value_counts()

Queue Name
Staff Directory English Transfer        11658
Clinic Voicemail Transfer               10177
Front Desk Transfer                      6480
Intake Outdial Queue                     3560
Family                                   2155
Consumer                                 1596
Housing                                  1173
Staff Directory Spanish Transfer         1103
Benefits                                 1083
Criminal Records Voicemail Transfer       876
SubSenior Family                          539
SubSenior Consumer                        492
Employment                                453
SubSenior Benefits                        406
SubSenior Tenant                          400
SubSenior ADAPT                           355
SubSenior Homeowner                       317
HIV Voicemail Transfer                    207
ADAPT                                     205
Family SP                                 167
Immigration                               122
SubSenior Other        

In [7]:
df_old['Queue Name'].value_counts()

Queue Name
Staff Directory English Transfer        96947
Clinic Voicemail Transfer               87747
Front Desk Transfer                     58217
Intake Outdial Queue                    42332
Family                                  16015
Staff Directory Spanish Transfer        13613
Consumer                                12250
Housing                                  8720
Criminal Records Voicemail Transfer      7764
SubSenior Other                          7606
Benefits                                 6825
SubSenior Tenant                         4662
SubSenior Benefits                       4396
Employment                               4337
SubSenior Consumer                       3503
SubSenior Family                         3131
ADAPT                                    3106
SubSenior Homeowner                      3103
HIV Voicemail Transfer                   2880
Family SP                                2388
SubSenior ADAPT                          1693
Safe Haven Transfer    

In [22]:
#Filter dataframe
# Steps, check all rows for an ID
# If all the Queue Names for that ID are either na or contain 'Transfer', drop that ID entirely
def filter_transfer_only(df):
    ids_to_drop = []
    for contact_id, group in df.groupby('Contact Session ID'):
        queue_names = group['Queue Name'].dropna().astype(str)
        if all('transfer' in name.lower() or 'intake outdial' in name.lower() for name in queue_names) or queue_names.empty:
            ids_to_drop.append(contact_id)
    filtered_df = df[~df['Contact Session ID'].isin(ids_to_drop)]
    return filtered_df

df_new_filt = filter_transfer_only(df_new)
df_old_filt = filter_transfer_only(df_old)

In [23]:
df_new.shape, df_new_filt.shape, df_old.shape, df_old_filt.shape

((331992, 9), (42103, 9), (3222287, 9), (410341, 9))

In [24]:
# First Step: Check if proportion of senior menu queues is the same in both datasets
# Filter out all queues that contain transfer

ids_senior_new = df_new_filt[df_new_filt['Queue Name'].str.contains('Senior', na=False)]['Contact Session ID'].unique()
ids_senior_old = df_old_filt[df_old_filt['Queue Name'].str.contains('Senior', na=False)]['Contact Session ID'].unique()

prop_senior_new = len(ids_senior_new) / df_new_filt['Contact Session ID'].nunique()
prop_senior_old = len(ids_senior_old) / df_old_filt['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")

prop new: 35.3860 %, prop old: 40.4776 %


In [18]:
temp = df_new_filt[df_new_filt['Contact Session ID'].isin(ids_senior_new)]
temp['Queue Name'].value_counts()

Queue Name
SubSenior Family           539
SubSenior Consumer         492
SubSenior Benefits         406
SubSenior Tenant           400
SubSenior ADAPT            355
SubSenior Homeowner        317
SubSenior Other            112
SubSenior Employment        76
SubSenior Consumer SP       71
SubSenior Family SP         65
SubSenior Benefits SP       34
SubSenior Homeowner SP      18
SubSenior Other SP          13
SubSenior Employment SP      8
Housing SubSeniors           8
SubSenior ADAPT SP           6
Name: count, dtype: int64